# Exercise

We are going to download CSV data from NOAA's website that gathers tornado data for both Texas and Oklahoma. We will then clean the data, select only the fields we are interested in, and load it into a SQLite database.

**STEP 1:** First import the necessary libraries. 

In [ ]:
import pandas as pd
import sqlite3

**STEP 2:** Import data from CSV

In [ ]:
df = pd.read_csv("storm_data.csv")
df

**STEP 3:** Extract out only the fields of interest.

In [ ]:
fields = ["CZ_NAME_STR","BEGIN_LOCATION","BEGIN_DATE","BEGIN_TIME","TOR_F_SCALE",
          "DEATHS_DIRECT","INJURIES_DIRECT","DAMAGE_PROPERTY_NUM","DAMAGE_CROPS_NUM",
          "STATE_ABBR","END_LOCATION","END_DATE","END_TIME",
          "EVENT_NARRATIVE","EPISODE_NARRATIVE"]

df.drop(columns=[col for col in df if col not in ?], inplace=True)


**STEP 4:** Convert date/time fields to a single datetime in new fields. Clean up the times so they have 4 digits and a colon. Then Convert those new fields to UTC. Finally, drop the original date/time fields.

In [ ]:
def clean_time(time):
    time_str = str(time).strip()
    c = f"{'0' * (4-len(time_str))}{time_str}"
    return c[0:2] + ":" + c[?:?]

df.insert(2, 'BEGIN_DATETIME', pd.to_datetime(df['BEGIN_DATE'] + ' ' + df['BEGIN_TIME'].apply(clean_time))  \
    .dt.tz_localize('US/Central') \
    .dt.tz_convert('UTC')
          )

df.insert(3, 'END_DATETIME', pd.to_datetime(df['END_DATE'] + ' ' + df['END_TIME'].apply(clean_time))  \
    .dt.tz_localize('US/Central') \
    .dt.tz_convert('UTC')
)

df.drop([?,?,?,?], axis=1, inplace=True)

df

**STEP 5:** Rename a fiew fields to make them easier to identify for end users.

In [ ]:
df.rename(columns= { 
    ? : "COUNTY_NAME",
    ? : "DAMAGE_PROPERTY_USD",
    ? : "DAMAGE_CROPS_USD"
})


**STEP 6:** Load the data into a SQLite database file, into a table called `TORNADO_TRACK`.

In [ ]:
conn = sqlite3.connect('my_database.db')
df.to_sql("TORNADO_TRACK", conn, if_exists='replace', index=False)

# 4. VERIFY DATA IS LOADED USING A SELECT query 
sql_df = pd.read_sql(?, conn)
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
  display(sql_df)

conn.close()